<a href="https://colab.research.google.com/github/netic43/-clean-token./blob/main/Chimera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import hashlib
import json
import time
import uuid
import requests
from flask import Flask, jsonify, request
from typing import List, Set

# --- SOLARA PROTOCOL CONFIGURATION ---
GENESIS_PREVIOUS_HASH = "0" * 64
BLOCK_REWARD = 100.0
# Difficulty: '00' is relatively easy for classical CPUs to 'collapse'
MAIN_DIFFICULTY_TARGET = "00"
ECHO_DIFFICULTY_TARGET = "0"

class Transaction:
    def __init__(self, sender: str, recipient: str, amount: float, signature: str = "DEMO_SIG"):
        self.sender = sender
        self.recipient = recipient
        self.amount = amount
        self.timestamp = time.time()
        self.signature = signature

    def to_dict(self):
        return {
            "sender": self.sender,
            "recipient": self.recipient,
            "amount": self.amount,
            "timestamp": self.timestamp,
            "signature": self.signature
        }

    def hash(self):
        return hashlib.sha256(json.dumps(self.to_dict(), sort_keys=True).encode()).hexdigest()

class Block:
    def __init__(self, index: int, transactions: List[Transaction], previous_hash: str, timestamp: float = None):
        self.index = index
        self.transactions = transactions
        self.previous_hash = previous_hash
        self.timestamp = timestamp or time.time()
        self.measurement_nonce = 0
        self.echo_nonce = 0
        self.qgs = self.calculate_qgs()
        self.hash = ""

    def calculate_qgs(self):
        """Quantum Genesis State: The unobserved state of the block."""
        block_raw = {
            "index": self.index,
            "transactions": [tx.to_dict() for tx in self.transactions],
            "previous_hash": self.previous_hash,
            "timestamp": self.timestamp
        }
        return hashlib.sha256(json.dumps(block_raw, sort_keys=True).encode()).hexdigest()

    def calculate_collapsed_hash(self):
        """The hash after observation (collapse)."""
        collapse_data = f"{self.qgs}{self.measurement_nonce}{self.echo_nonce}"
        return hashlib.sha256(collapse_data.encode()).hexdigest()

class SolaraNode:
    def __init__(self):
        self.chain: List[Block] = []
        self.pending_transactions: List[Transaction] = []
        self.peers: Set[str] = set()
        self.node_id = str(uuid.uuid4()).replace('-', '')

        # Create Genesis Block
        self.create_genesis()

    def create_genesis(self):
        genesis = Block(0, [], GENESIS_PREVIOUS_HASH)
        genesis.hash = genesis.calculate_collapsed_hash()
        self.chain.append(genesis)

    @property
    def last_block(self):
        return self.chain[-1]

    def add_transaction(self, sender, recipient, amount):
        tx = Transaction(sender, recipient, amount)
        self.pending_transactions.append(tx)
        return self.last_block.index + 1

    def proof_of_collapse(self, block: Block):
        """
        Emulates the Quantum Observation Device (QOD) loop.
        Requires finding two separate nonces that satisfy the Solara Protocol.
        """
        print(f"[*] QOD initiating collapse for Block {block.index}...")

        # Phase 1: Measurement Collapse
        while not hashlib.sha256(f"{block.qgs}{block.measurement_nonce}".encode()).hexdigest().startswith(MAIN_DIFFICULTY_TARGET):
            block.measurement_nonce += 1

        # Phase 2: Multi-Versal Echo
        while not hashlib.sha256(f"{block.qgs}{block.echo_nonce}".encode()).hexdigest().startswith(ECHO_DIFFICULTY_TARGET):
            block.echo_nonce += 1

        block.hash = block.calculate_collapsed_hash()
        print(f"[+] Block {block.index} successfully collapsed. Hash: {block.hash[:10]}...")
        return block

    def valid_chain(self, chain: List[Block]):
        """Consensus: Validate an incoming chain from a peer."""
        last_block = chain[0]
        current_index = 1
        while current_index < len(chain):
            block = chain[current_index]
            # Check link
            if block.previous_hash != last_block.hash:
                return False
            # Check Collapse Proof
            if not hashlib.sha256(f"{block.qgs}{block.measurement_nonce}".encode()).hexdigest().startswith(MAIN_DIFFICULTY_TARGET):
                return False
            last_block = block
            current_index += 1
        return True

    def resolve_conflicts(self):
        """Longest chain consensus rule."""
        new_chain = None
        max_length = len(self.chain)

        for peer in self.peers:
            response = requests.get(f"http://{peer}/chain")
            if response.status_code == 200:
                length = response.json()['length']
                chain = response.json()['chain']
                # (In production, we'd deserialize JSON back to objects)
                if length > max_length: # simplified for demo
                    max_length = length
                    new_chain = chain

        if new_chain:
            # self.chain = new_chain
            return True
        return False

# --- API LAYER ---
app = Flask(__name__)
solara = SolaraNode()

@app.route('/transactions/new', methods=['POST'])
def new_transaction():
    values = request.get_json()
    required = ['sender', 'recipient', 'amount']
    if not all(k in values for k in required):
        return 'Missing values', 400
    index = solara.add_transaction(values['sender'], values['recipient'], values['amount'])
    return jsonify({'message': f'Transaction will be added to Block {index}'}), 201

@app.route('/mine', methods=['GET'])
def mine():
    # Reward for the node (Observer Reward)
    solara.add_transaction(sender="SOLARA_NETWORK", recipient=solara.node_id, amount=BLOCK_REWARD)

    last_block = solara.last_block
    new_block = Block(len(solara.chain), solara.pending_transactions, last_block.hash)

    # Run PoC
    collapsed_block = solara.proof_of_collapse(new_block)

    solara.chain.append(collapsed_block)
    solara.pending_transactions = []

    return jsonify({
        'message': "New Block Collapsed",
        'index': collapsed_block.index,
        'hash': collapsed_block.hash,
        'measurement_nonce': collapsed_block.measurement_nonce,
        'echo_nonce': collapsed_block.echo_nonce
    }), 200

@app.route('/chain', methods=['GET'])
def full_chain():
    response = {
        'chain': [vars(b) for b in solara.chain],
        'length': len(solara.chain),
    }
    return jsonify(response), 200

@app.route('/nodes/register', methods=['POST'])
def register_nodes():
    values = request.get_json()
    nodes = values.get('nodes')
    for node in nodes:
        solara.peers.add(node)
    return jsonify({'message': 'New nodes added', 'total_nodes': list(solara.peers)}), 201

if __name__ == '__main__':
    from argparse import ArgumentParser
    parser = ArgumentParser()
    parser.add_argument('-p', '--port', default=5000, type=int, help='port to listen on')
    args = parser.parse_args()

    print(f"--- [SOLARA GENESIS NODE LAUNCHED] ---")
    print(f"Node ID: {solara.node_id}")
    app.run(host='0.0.0.0', port=args.port)